### Notes

There are multiple approaches to this problem. I'm starting off with a resnet w/ FPN, but there are probably other models we could replace it with. This is to get a baseline up and running quickly.

Of note, there are some instances here where I basically just repeat some code snippets due to laziness/prioritization. Refactoring this would be something to consider if we were to work more on this, in addition to notebooks being kind of a pain to work with once they get large enough.

I've tested a variation of this notebook on IDUN clusters and it works well, especially with the `Python/3.10.8-GCCcore-12.2.0` module loaded.

In [ ]:
import torch
import torchvision
import random
import numpy as np
import cv2 # Added cv2 import
import os # Added os import

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [ ]:
import os

# NOTE: Update this path if necessary
#base_folder = '/cluster/home/sebassm/tdt4265' # Was used for idun
base_folder = r"C:\Users\sebbe\code\skole\TDT4265\project\lidar\lidar"
train_img_dir = os.path.join(base_folder, 'images/train')
val_img_dir = os.path.join(base_folder, 'images/valid')
train_label_dir = os.path.join(base_folder, 'labels/train')
val_label_dir = os.path.join(base_folder, 'labels/valid')

# Verify paths exist (optional but recommended)
assert os.path.isdir(train_img_dir), f"Training image directory not found: {train_img_dir}"
assert os.path.isdir(val_img_dir), f"Validation image directory not found: {val_img_dir}"
assert os.path.isdir(train_label_dir), f"Training label directory not found: {train_label_dir}"
assert os.path.isdir(val_label_dir), f"Validation label directory not found: {val_label_dir}"

In [ ]:
from torch.utils.data import Dataset
import cv2
import torch
import os
import numpy as np
import torchvision # Added for fallback transform

class SnowPoleDataset(Dataset):
    def __init__(self, img_dir, label_dir, transforms=None):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.transforms = transforms
        # Get sorted list of image filenames (assuming common image extensions)
        self.image_files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.img_dir, img_name)
        
        # Construct label path
        label_name = os.path.splitext(img_name)[0] + '.txt'
        label_path = os.path.join(self.label_dir, label_name)
        
        # Load image
        img = cv2.imread(img_path)
        if img is None:
            # Return None if image cannot be read, DataLoader's collate_fn should handle this
            print(f"Warning: Could not read image: {img_path}. Skipping.")
            return None, None 
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_h, img_w = img.shape[:2]
        
        # Load labels (YOLO format: class x_center y_center width height - normalized)
        boxes = []
        class_labels = []
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) == 5:
                        class_id, x_center, y_center, width, height = map(float, parts)
                        
                        # Convert YOLO format (normalized) to Pascal VOC (absolute pixels)
                        x_center_abs = x_center * img_w
                        y_center_abs = y_center * img_h
                        width_abs = width * img_w
                        height_abs = height * img_h
                        
                        xmin = x_center_abs - (width_abs / 2)
                        ymin = y_center_abs - (height_abs / 2)
                        xmax = x_center_abs + (width_abs / 2)
                        ymax = y_center_abs + (height_abs / 2)
                        
                        # Clamp coordinates to image bounds
                        xmin = max(0, xmin)
                        ymin = max(0, ymin)
                        xmax = min(img_w, xmax)
                        ymax = min(img_h, ymax)
                        
                        # Ensure box has valid dimensions
                        if xmax > xmin and ymax > ymin:
                           boxes.append([xmin, ymin, xmax, ymax])
                           # Assuming class_id 0 in YOLO corresponds to label 1 (snow pole)
                           class_labels.append(int(class_id) + 1) 
        
        # Convert to initial tensors
        boxes_tensor = torch.as_tensor(boxes, dtype=torch.float32)
        if not boxes:
             boxes_tensor = torch.zeros((0, 4), dtype=torch.float32)
        labels_tensor = torch.as_tensor(class_labels, dtype=torch.int64)

        target = {} # Initialize target dict
        target['boxes'] = boxes_tensor
        target['labels'] = labels_tensor
        # Store image_id as a plain Python integer
        target['image_id'] = int(idx)
        if boxes_tensor.shape[0] > 0:
            area = (boxes_tensor[:, 2] - boxes_tensor[:, 0]) * (boxes_tensor[:, 3] - boxes_tensor[:, 1])
            target['area'] = area
        else:
            target['area'] = torch.zeros((0,), dtype=torch.float32)
        target['iscrowd'] = torch.zeros((boxes_tensor.shape[0],), dtype=torch.int64) # Assume no crowd

        if self.transforms:
            # Prepare data for Albumentations: combine boxes and labels
            bboxes_for_transform = []
            if target['boxes'].shape[0] > 0:
                # Ensure labels are correctly shaped for concatenation
                labels_for_concat = target['labels'].unsqueeze(1).float() # Needs to be float for concat
                # Combine boxes [N, 4] and labels [N, 1] -> [N, 5]
                combined = torch.cat((target['boxes'], labels_for_concat), dim=1)
                bbox_with_class = combined.tolist()

            try:
                # Apply transformations to both image and bounding boxes
                transformed = self.transforms(image=img, bboxes=bbox_with_class)
                img = transformed['image']
                
                # Extract transformed boxes+labels
                transformed_bboxes = transformed['bboxes']
                if transformed_bboxes:
                    # Convert back to tensors
                    bbox_tensor = torch.as_tensor(transformed_bboxes, dtype=torch.float32)
                    target['boxes'] = bbox_tensor[:, :4]
                    target['labels'] = bbox_tensor[:, 4].to(torch.int64)
                    # Update area based on new boxes
                    target['area'] = (target['boxes'][:, 2] - target['boxes'][:, 0]) * \
                                    (target['boxes'][:, 3] - target['boxes'][:, 1])
                else:
                    # All boxes were removed by the transform
                    target['boxes'] = torch.zeros((0, 4), dtype=torch.float32)
                    target['labels'] = torch.zeros((0,), dtype=torch.int64)
                    target['area'] = torch.zeros((0,), dtype=torch.float32)
                target['iscrowd'] = torch.zeros((target['boxes'].shape[0],), dtype=torch.int64)
                
            except (ValueError, TypeError) as e:
                # Handle transformation failures gracefully
                print(f"Transform failed for {img_path}: {e}")
                print(f"Box data that caused the error: {target['boxes']}")
                
                # Fall back to basic tensor conversion
                try:
                    img = torchvision.transforms.functional.to_tensor(img)
                except Exception:
                    print(f"Failed to convert image to tensor - skipping sample")
                    return None, None
                
                # Keep original annotations
                target['boxes'] = target['boxes'].to(torch.float32)
                target['labels'] = target['labels'].to(torch.int64)
                if target['boxes'].shape[0] == 0:
                    target['area'] = torch.zeros((0,), dtype=torch.float32)
                    target['iscrowd'] = torch.zeros((0,), dtype=torch.int64)
                else:
                     target['area'] = (target['boxes'][:, 2] - target['boxes'][:, 0]) * \
                                     (target['boxes'][:, 3] - target['boxes'][:, 1])
                     target['iscrowd'] = torch.zeros((target['boxes'].shape[0],), dtype=torch.int64)
                print("Using original boxes with basic tensor conversion")

        # Ensure correct tensor types
        target['boxes'] = target['boxes'].to(torch.float32)
        target['labels'] = target['labels'].to(torch.int64)

        # Sanity check before returning
        if not isinstance(target.get('boxes'), torch.Tensor) or not isinstance(target.get('labels'), torch.Tensor):
            print(f"Invalid annotation format in {img_path} - skipping")
            return None, None
            
        return img, target

    def __len__(self):
        return len(self.image_files)


In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

train_transform = A.Compose([
    A.RandomBrightnessContrast(p=0.3),
    A.Rotate(limit=5, p=0.5, border_mode=cv2.BORDER_CONSTANT),
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
], bbox_params=A.BboxParams(format='pascal_voc'))

val_transform = A.Compose([
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
], bbox_params=A.BboxParams(format='pascal_voc'))

In [ ]:
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

def get_model(num_classes):
    # Load a model pre-trained on COCO
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=torchvision.models.detection.FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
    
    # Get the number of input features for the classifier
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    # Replace the pre-trained head with a new one
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    
    return model

# num_classes includes background, so 1 class (snow pole) + background = 2
model = get_model(num_classes=2)


In [ ]:
# Install pycocotools for evaluation. We still need a couple of files in working dir to run the evaluation code.
%pip install pycocotools --quiet

In [ ]:
# Import evaluation utilities
# NOTE: We are using some local python files here for evaluation, such as engine.py and utils.py (or coco_utils.py).
# These files are not part of the standard library and need to be in the working directory.

try:
    # Attempt to import from a local copy or installed reference scripts
    from engine import evaluate 
    import utils # coco_utils aliased as utils
    print("Successfully imported evaluation utilities.")
except ImportError:
    print("WARNING: Could not import evaluation utilities (engine.py, utils.py/coco_utils.py). ")
    print("Evaluation step will be skipped. Please ensure these files are available.")
    # Define a dummy evaluate function to avoid errors later
    def evaluate(model, data_loader, device):
        print("Evaluation skipped: Utilities not found.")
        return None # Or return dummy stats

In [ ]:
from torch.utils.data import DataLoader

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
num_epochs = 10 # Set number of epochs
batch_size = 16 # Adjust batch size based on GPU memory
num_workers = 0 # Adjust based on system

# --- Instantiate Datasets ---
train_dataset = SnowPoleDataset(img_dir=train_img_dir, label_dir=train_label_dir, transforms=train_transform)
val_dataset = SnowPoleDataset(img_dir=val_img_dir, label_dir=val_label_dir, transforms=val_transform)

# --- Define Collate Function ---
def collate_fn(batch):
    # Filter out samples where __getitem__ returned (None, None)
    batch = [(img, tgt) for img, tgt in batch if img is not None and tgt is not None]
    if not batch: # If the whole batch is invalid
        # Return empty tuples to signify an empty batch
        return ((), ()) 
    return tuple(zip(*batch))

# --- Instantiate DataLoaders ---
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=num_workers, collate_fn=collate_fn)
# --- End Data Loading ---

# Optimizer here
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005,
                            momentum=0.9, weight_decay=0.0005)

# Also learnin rate
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer,
                                               step_size=15, # Adjust step size
                                               gamma=0.1) # Adjust gamma

print(f"Starting Training on {device}...")
print(f"Training samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")

train_losses = []
val_map_50_95 = []
val_map_50 = []
val_ar = []
val_p = []

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    batches_processed = 0 # Keep track of batches
    
    # --- Training Loop ---
    for i, batch_data in enumerate(train_loader): 
        # Check if collate_fn returned indication of an empty batch
        # We don't want to process empty batches
        if batch_data == ((), ()):
            print(f"Skipping batch {i} due to empty batch after filtering.")
            continue
            
        images, targets = batch_data
        
        # Ensure images and targets are lists/tuples (collate_fn returns tuple of tuples)
        if not isinstance(images, (list, tuple)) or not isinstance(targets, (list, tuple)):
             print(f"Skipping batch {i} due to unexpected data format from collate_fn.")
             continue
             
        images = list(image.to(device) for image in images)
        # Move only tensor values in targets to the device, sorry for this botched reading experience of the code line below
        targets = [{k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in t.items()} for t in targets]

        # Check for empty targets
        valid_targets = []
        valid_indices = []
        for idx, t in enumerate(targets):
            # Check if 'boxes' key exists and has items
            if 'boxes' in t and t['boxes'].shape[0] > 0:
                valid_targets.append(t)
                valid_indices.append(idx)
                
        if not valid_targets:
            print(f"Skipping batch {i} due to empty targets after transformations or errors.")
            continue
            
        # Filter images corresponding to valid targets
        images = [images[idx] for idx in valid_indices]
        # Targets are already filtered
        targets = valid_targets 

        # Faster R-CNN returns a dict of losses in train mode
        try:
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
        except Exception as model_err:
            print(f"Error during model forward/loss calculation for batch {i}: {model_err}")
            # Optionally print shapes or inspect data, but this might break something if the data is malformed
            print(f"  Image shapes: {[img.shape for img in images]}")
            print(f"  Target sample: {targets[0] if targets else 'No targets'}")
            continue # Skip this batch

        # Check for NaN or inf losses
        if not torch.isfinite(losses):
            print(f"WARNING: Non-finite loss detected: {losses.item()}. Skipping batch {i}.")
            print(f"Loss dict: {loss_dict}")
            continue

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        epoch_loss += losses.item()
        batches_processed += 1 # Increment successful batch count
        
        if (i + 1) % 50 == 0: # Print progress every 50 batches, similar to yolo but not continuous
             print(f"  Batch {i+1}/{len(train_loader)}, Batch Loss: {losses.item():.4f}")
    # --- End Training Loop ---
    
    # Update the learning rate
    lr_scheduler.step()

    # Calculate average loss based on successfully processed batches
    if batches_processed > 0:
        avg_epoch_loss = epoch_loss / batches_processed
        train_losses.append(avg_epoch_loss)
        print(f"Epoch {epoch+1}/{num_epochs}, Average Training Loss: {avg_epoch_loss:.4f}")
    else:
        train_losses.append(float('nan')) # Append NaN if no batches were processed
        print(f"Epoch {epoch+1}/{num_epochs}, No batches processed successfully in training.")

    print(f"Running Validation for Epoch {epoch+1}...")
    current_map_50_95 = 0.0 # Default values if evaluation fails
    current_map_50 = 0.0
    current_ar = 0.0
    current_p = 0.0
    if 'evaluate' in globals() and callable(evaluate): # Check if evaluate is defined and callable
        # Ensure model is in evaluation mode for validation
        model.eval()
        coco_evaluator = evaluate(model, val_loader, device=device)
        if coco_evaluator and hasattr(coco_evaluator, 'coco_eval') and 'bbox' in coco_evaluator.coco_eval:
            stats = coco_evaluator.coco_eval['bbox'].stats
            current_map_50_95 = stats[0] # AP@0.50:0.95
            current_map_50 = stats[1]    # AP@0.50
            current_ar = stats[8]       # AR@0.50:0.95
            print(f"Validation Epoch {epoch+1} - mAP@0.50:0.95: {current_map_50_95:.4f}, mAP@0.50: {current_map_50:.4f}")
        else:
            print("Validation evaluation did not produce expected results.")
    else:
        print("Skipping validation because evaluate function is not available or callable.")

    val_map_50_95.append(current_map_50_95)
    val_map_50.append(current_map_50)
    val_ar.append(current_ar)
    

print("Training Finished.")

# --- Run Final Validation After Training ---
print("Running Final Validation...")
if 'evaluate' in globals():
    coco_evaluator = evaluate(model, val_loader, device=device)
    if coco_evaluator:
        stats = coco_evaluator.coco_eval['bbox'].stats
        print(f"Validation AP@0.50:0.95: {stats[0]:.4f}")
        print(f"Validation AP@0.50:      {stats[1]:.4f}")
        # Print other stats if desired
        # print(f"Validation AP@0.75:      {stats[2]:.4f}")
        # print(f"Validation AP (small):   {stats[3]:.4f}")
        # print(f"Validation AP (medium):  {stats[4]:.4f}")
        # print(f"Validation AP (large):   {stats[5]:.4f}")
else:
    print("Skipping final validation because evaluate function is not defined.")
# --- End Final Validation ---

In [ ]:
# --- Save the Model ---
model_save_path = os.path.join(base_folder, 'fasterrcnn_snowpole_model.pth')
print(f"Saving model state dictionary to: {model_save_path}")
torch.save(model.state_dict(), model_save_path)
print("Model saved successfully.")

## Model Evaluation

The training loop now includes a validation step after each epoch using the `evaluate` function (leveraging `pycocotools`). Standard COCO metrics (like Average Precision) are printed for the validation set.

In [ ]:
# This cell remains empty or can be used for saving the model, etc.

In [ ]:
# Getting precision and recall from the coco_evaluator using `accumulate()` method
if 'coco_evaluator' in globals() and coco_evaluator:
    coco_evaluator.accumulate()
    coco_evaluator.summarize()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np 

print("Generating training plots...")

epochs = range(1, num_epochs + 1)

# Filter out potential NaN values for plotting if any epoch had zero batches
valid_train_indices = ~np.isnan(train_losses)
valid_epochs_train = np.array(epochs)[valid_train_indices]
valid_train_losses = np.array(train_losses)[valid_train_indices]

plt.figure(figsize=(18, 5))

# Plot Training Loss
plt.subplot(1, 3, 1) 
if len(valid_epochs_train) > 0:
    plt.plot(valid_epochs_train, valid_train_losses, 'bo-', label='Training Loss')
    plt.title('Training Loss per Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
else:
    plt.text(0.5, 0.5, 'No valid training data to plot', horizontalalignment='center', verticalalignment='center')


# Plot Validation mAP
plt.subplot(1, 3, 2)
plt.plot(epochs, val_map_50_95, 'ro-', label='mAP@0.50:0.95')
plt.plot(epochs, val_map_50, 'go-', label='mAP@0.50')
plt.title('Validation mAP per Epoch')
plt.xlabel('Epoch')
plt.ylabel('mAP')
plt.legend()
plt.grid(True)

# Plot Validation Average Recall
plt.subplot(1, 3, 3)
plt.plot(epochs, val_ar, 'mo-', label='AR@100')
plt.title('Validation Average Recall per Epoch')
plt.xlabel('Epoch')
plt.ylabel('AR')
plt.legend()
plt.grid(True)


plt.tight_layout()
plt.show()

print("Plotting finished.")

In [ ]:
# This code is just for loading the model and running inference without training it again
# It does not really need to be run in the same script as the training code, but is included here for completeness


from codecarbon import EmissionsTracker
# Initialize the tracker
tracker = EmissionsTracker() # Set country code for Norway
tracker.start()

model_save_path = os.path.join(base_folder, 'fasterrcnn_snowpole_model.pth')
model = get_model(num_classes=2) # Reinitialize model
model.load_state_dict(torch.load(model_save_path))
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

print(f"Initializing existing model from: {model_save_path}")
print("Device: ", device)


In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np
from albumentations.pytorch import ToTensorV2
import albumentations as A

# --- Load a single image for inference ---
# NOTE: Replace 'path/to/your/test_image.png' with an actual image path
img_path = os.path.join(val_img_dir, 'image_16.png')

try:
    img_display = cv2.imread(img_path)
    if img_display is None:
        raise FileNotFoundError(f"Could not read image file: {img_path}")
    img_display = cv2.cvtColor(img_display, cv2.COLOR_BGR2RGB)
except FileNotFoundError as e:
    print(e)
    print("--- Using a dummy white image for visualization structure check --- ")
    img_display = np.ones((512, 512, 3), dtype=np.uint8) * 255 # White dummy image here as fallback
    img_tensor = torch.rand(3, 512, 512).to(device) # Dummy tensor as well
else:
    # Define the same normalization transform used during validation/training
    # No augmentations like flips/rotations here, just normalization and ToTensor
    inference_transform = A.Compose([
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])
    # Apply the transform before inference
    img_tensor = inference_transform(image=img_display)['image'].to(device)


# --- Inference ---
model.eval()
with torch.no_grad():
    prediction = model([img_tensor])[0]

print("Prediction:", prediction)

# Draw prediction boxes on the display image
detection_threshold = 0.5 # Lowered threshold slightly just to see boxes if not too confident
img_with_boxes = img_display.copy()

for box, score, label in zip(prediction['boxes'], prediction['scores'], prediction['labels']):
    if score > detection_threshold:
        # Convert box coordinates to integers, just some casting here
        xmin, ymin, xmax, ymax = map(int, box.cpu().numpy())
        # pick a color (green if it’s our snow pole, red otherwise)
        color = (0, 255, 0) if label == 1 else (255, 0, 0)
        # slap a box around the detection
        cv2.rectangle(img_with_boxes, (xmin, ymin), (xmax, ymax), color, 2)
        # throw on the label and score
        text = f"Label: {label.item()} Score: {score.item():.2f}"
        cv2.putText(img_with_boxes, text, (xmin, ymin - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

# Display the image with boxes
plt.figure(figsize=(10, 10))
plt.imshow(img_with_boxes)
plt.axis('off')
plt.show()


In [ ]:
import os
import cv2
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2
import time

# --- Configuration ---
test_img_dir = os.path.join(base_folder, 'images/test')
results_dir = os.path.join(base_folder, 'results', 'faster_rcnn_run')
detection_threshold = 0.5 # Confidence threshold for saving predictions
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
if device.type == 'cuda':
    device = torch.device("cuda")

# Ensure results directory exists
os.makedirs(results_dir, exist_ok=True)

# Define the inference transform (same as validation)
inference_transform = A.Compose([
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

# --- Prediction Loop ---
model.to(device) # Ensure model is on the correct device
model.eval()

print(f"Starting prediction on test images in: {test_img_dir}")
print(f"Saving results to: {results_dir}")

test_image_files = sorted([f for f in os.listdir(test_img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

# --- FPS Tracking Initialization ---
start_time = time.time()
images_processed_count = 0
# ---

with torch.no_grad():
    for i, img_name in enumerate(test_image_files):
        img_path = os.path.join(test_img_dir, img_name)

        # Load image
        img = cv2.imread(img_path)
        if img is None:
            print(f"Warning: Could not read image {img_path}. Skipping.")
            continue
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        H, W = img.shape[:2]

        # Apply transform
        transformed = inference_transform(image=img_rgb)
        img_tensor = transformed['image'].unsqueeze(0).to(device)

        # --- Measure Inference Time ---
        # iter_start_time = time.time()

        # Get prediction
        prediction = model(img_tensor)[0]

        # iter_end_time = time.time()
        # print(f"  Inference time for {img_name}: {iter_end_time - iter_start_time:.4f}s")
        # ---

        # Prepare output file path
        txt_filename = os.path.splitext(img_name)[0] + '.txt'
        txt_path = os.path.join(results_dir, txt_filename)

        # Process and save predictions
        with open(txt_path, 'w') as f:
            for box, score, label in zip(prediction['boxes'], prediction['scores'], prediction['labels']):
                if score > detection_threshold:
                    # Convert box from Pascal VOC (xmin, ymin, xmax, ymax) to YOLO format
                    xmin, ymin, xmax, ymax = box.cpu().numpy()

                    # Clamp coordinates to image bounds (important after model prediction)
                    xmin = max(0, xmin)
                    ymin = max(0, ymin)
                    xmax = min(W, xmax)
                    ymax = min(H, ymax)

                    if xmax <= xmin or ymax <= ymin: # Skip invalid boxes
                        continue

                    box_w = xmax - xmin
                    box_h = ymax - ymin
                    x_center = xmin + box_w / 2
                    y_center = ymin + box_h / 2

                    # Normalize
                    x_center_norm = x_center / W
                    y_center_norm = y_center / H
                    width_norm = box_w / W
                    height_norm = box_h / H

                    # Adjust class ID (Faster R-CNN uses 1 for snow pole)
                    class_id = label.item() - 1
                    confidence = score.item()

                    # Write line: <class_id> <x_center_norm> <y_center_norm> <width_norm> <height_norm> <confidence>
                    f.write(f"{class_id} {x_center_norm:.6f} {y_center_norm:.6f} {width_norm:.6f} {height_norm:.6f} {confidence:.6f}\n")

        images_processed_count += 1 # Increment counter for successfully processed images

        if (i + 1) % 50 == 0:
            print(f"  Processed {i+1}/{len(test_image_files)} images...")

# --- FPS Calculation ---
end_time = time.time()
total_time = end_time - start_time
fps = 0
if total_time > 0:
    fps = images_processed_count / total_time
# ---

print(f"Prediction finished.")
print(f"Processed {images_processed_count} images in {total_time:.2f} seconds.")
print(f"Average Inference Speed: {fps:.2f} FPS")


In [ ]:
# Stop the emissions tracker
tracker.stop()
# Get the total emissions